# 02 — Resistance Sweep Analysis

This notebook analyzes the cleaned resistance sweep data descriptively:
- all gain curves
- source loading
- peak gain
- resonance frequency
- approximate bandwidth
- approximate Q factor

It should not fit parameters yet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/processed/resistance_sweep_combined_clean.csv")

PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures/resistance_sweep")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
})

In [ ]:
df = pd.read_csv(DATA_PATH)

required = ["Resistance", "Frequency_Hz", "Vs_Vpp", "VR_Vpp", "Gain"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Current columns: {df.columns.tolist()}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required)
df = df.sort_values(["Resistance", "Frequency_Hz"]).reset_index(drop=True)

df.head()

In [ ]:
# Average repeated runs at each resistance-frequency point for descriptive curves.
df_mean = (
    df.groupby(["Resistance", "Frequency_Hz"], as_index=False)
    .agg(
        Vs_Vpp=("Vs_Vpp", "mean"),
        VR_Vpp=("VR_Vpp", "mean"),
        Gain=("Gain", "mean"),
        Gain_Std=("Gain", "std"),
        N=("Gain", "count"),
    )
)

df_mean["Gain_Std"] = df_mean["Gain_Std"].fillna(0)

df_mean.to_csv(PROCESSED_DIR / "resistance_sweep_mean_curves.csv", index=False)
df_mean.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for R, group in df_mean.groupby("Resistance"):
    group = group.sort_values("Frequency_Hz")

    ax.semilogx(
        group["Frequency_Hz"],
        group["Gain"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=f"{R:.0f} Ω"
    )

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Gain |VR/Vs|")
ax.set_title("Frequency Response Under Varying Resistive Damping")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Measured R")

fig.savefig(FIG_DIR / "resistance_sweep_gain_curves.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "resistance_sweep_gain_curves.svg", bbox_inches="tight")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for R, group in df_mean.groupby("Resistance"):
    group = group.sort_values("Frequency_Hz")

    ax.semilogx(
        group["Frequency_Hz"],
        group["Vs_Vpp"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=f"{R:.0f} Ω"
    )

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Measured Source Voltage Vs (Vpp)")
ax.set_title("Source Loading Across Resistance Sweep")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Measured R")

fig.savefig(FIG_DIR / "source_loading_resistance_sweep.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "source_loading_resistance_sweep.svg", bbox_inches="tight")

plt.show()

In [ ]:
def summarize_curve(group):
    group = group.sort_values("Frequency_Hz")

    f = group["Frequency_Hz"].values
    g = group["Gain"].values
    vs = group["Vs_Vpp"].values

    peak_gain = np.max(g)
    peak_indices = np.where(np.isclose(g, peak_gain))[0]

    peak_freq_low = f[peak_indices[0]]
    peak_freq_high = f[peak_indices[-1]]
    peak_freq_mid = 0.5 * (peak_freq_low + peak_freq_high)

    half_power_gain = peak_gain / np.sqrt(2)
    above = g >= half_power_gain

    if np.sum(above) >= 2:
        f_low = f[above][0]
        f_high = f[above][-1]
        bandwidth = f_high - f_low
        q_approx = peak_freq_mid / bandwidth if bandwidth > 0 else np.nan
    else:
        f_low = np.nan
        f_high = np.nan
        bandwidth = np.nan
        q_approx = np.nan

    vs_baseline = vs[0]
    vs_min = np.min(vs)
    vs_drop = vs_baseline - vs_min
    vs_drop_percent = 100 * vs_drop / vs_baseline

    return pd.Series({
        "Peak_Gain": peak_gain,
        "Peak_Frequency_Hz": peak_freq_mid,
        "Peak_Frequency_Low_Hz": peak_freq_low,
        "Peak_Frequency_High_Hz": peak_freq_high,
        "Half_Power_Gain": half_power_gain,
        "f_low_Hz": f_low,
        "f_high_Hz": f_high,
        "Bandwidth_Hz": bandwidth,
        "Q_Approx": q_approx,
        "Vs_Baseline_Vpp": vs_baseline,
        "Vs_Min_Vpp": vs_min,
        "Vs_Drop_Vpp": vs_drop,
        "Vs_Drop_Percent": vs_drop_percent,
    })

summary = (
    df_mean.groupby("Resistance")
    .apply(summarize_curve)
    .reset_index()
)

summary.to_csv(PROCESSED_DIR / "resistance_sweep_summary.csv", index=False)
summary

In [ ]:
def save_line_plot(x, y, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, y, marker="o", linewidth=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.35)

    fig.savefig(FIG_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.svg", bbox_inches="tight")
    plt.show()

save_line_plot(summary["Resistance"], summary["Peak_Gain"],
               "Measured Resistance R (Ω)", "Peak Gain",
               "Peak Gain vs Resistance", "peak_gain_vs_resistance")

save_line_plot(summary["Resistance"], summary["Peak_Frequency_Hz"],
               "Measured Resistance R (Ω)", "Approximate Resonance Frequency (Hz)",
               "Resonance Frequency vs Resistance", "resonance_frequency_vs_resistance")

save_line_plot(summary["Resistance"], summary["Bandwidth_Hz"],
               "Measured Resistance R (Ω)", "Approximate Bandwidth (Hz)",
               "Bandwidth vs Resistance", "bandwidth_vs_resistance")

save_line_plot(summary["Resistance"], summary["Q_Approx"],
               "Measured Resistance R (Ω)", "Approximate Q Factor",
               "Q Factor vs Resistance", "q_factor_vs_resistance")

save_line_plot(summary["Resistance"], summary["Vs_Drop_Percent"],
               "Measured Resistance R (Ω)", "Source Voltage Drop (%)",
               "Source Loading vs Resistance", "source_voltage_drop_vs_resistance")

## Interpretation

Peak gain increases with measured resistance because a larger fraction of the series voltage appears across the measured resistor.

Source loading is strongest at low resistance because the circuit draws more current near resonance.

Bandwidth and Q estimates are approximate because they depend on frequency spacing. Do not oversell them as precise.